In [1]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

dir = Path().resolve().parents[1]

if dir not in sys.path:
    sys.path.append(str(dir))

Failed to read module file 'C:\Users\User\anaconda3\Lib\re\_casefix.py' for module 're._casefix': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\CodingHenry\research_MBKM\venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\CodingHenry\research_MBKM\venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1140, in _find_and_load_unlocked
ModuleNotFoundError: No mod

In [2]:
import joblib
from diffusion import reverse
from config import Config
import torch
from nn import Unet1D
from utils import posterior_beta, inverse_standard, log_transform, load_and_split_snp500
import time
import yfinance as yf
import math

In [3]:
data = load_and_split_snp500(transform_fn=log_transform)

In [4]:
train_snp500 = data["train"]

In [5]:
MODEL_PATH = dir / "models"
DIR_128 = MODEL_PATH / "model_128.pth"
DIR_256 = MODEL_PATH / "model_256.pth"
DIR_512 = MODEL_PATH / "model_512.pth"
assert [dir.exists() for dir in [DIR_128, DIR_256, DIR_512]], "model doesnt exist"

# SYN_PATH = dir / "data" / "syn_data_256.joblib"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device

'cpu'

In [6]:
T = 1000

config = Config().set_model_config(
  attn_res=16,
  n_res_block=2,
  T=T,
  num_heads=4,
  encoder_in_channels=[1, 4, 8, 16],
  encoder_out_channels=[4, 8, 16, 32],
  decoder_in_channels=[32, 16, 8, 4],
  decoder_out_channels=[16, 8, 4, 1]
)

betas = torch.linspace(1e-4, 2e-2, T)
alpha_hats = torch.cumprod(
  input=1-betas,
  dim=0,
  dtype=torch.float32
)
posterior_betas = torch.tensor([posterior_beta(alpha_hats=alpha_hats, betas=betas, t=t) for t in range(T)])

xT_128 = torch.randn(size=(1000, 1, 128))
xT_256 = torch.randn(size=(1000, 1, 256))
xT_512 = torch.randn(size=(1000, 1, 512))

In [7]:
model_128 = Unet1D(**config.model_config)
model_256 = Unet1D(**config.model_config)
model_512 = Unet1D(**config.model_config)

checkpoint_128 = torch.load(DIR_128, weights_only=True)
checkpoint_256 = torch.load(DIR_256, weights_only=True)
checkpoint_512 = torch.load(DIR_512, weights_only=True)

model_128.load_state_dict(checkpoint_128["model_state_dict"])
model_256.load_state_dict(checkpoint_256["model_state_dict"])
model_512.load_state_dict(checkpoint_512["model_state_dict"])

<All keys matched successfully>

In [8]:
reverse_time_128 = time.time()

scaled_synthetic_data_128 = reverse(
  xT=xT_128,
  T=T,
  betas=betas,
  posterior_betas=posterior_betas,
  alpha_bars=alpha_hats,
  model=model_128
).squeeze(1).detach().numpy()

reverse_time_128 = time.time() - reverse_time_128
print(f"Reverse process duration : { reverse_time_128:.2f} seconds")

synthetic_data_128 = inverse_standard(scaled_synthetic_data_128, train_snp500)
SYN_PATH_128 = dir / "data" / "syn_data_128.joblib"
joblib.dump(synthetic_data_128, SYN_PATH_128)

Reverse process duration : 130.07 seconds


['D:\\CodingHenry\\research_MBKM\\data\\syn_data_128.joblib']

In [9]:
reverse_time_256 = time.time()

scaled_synthetic_data_256 = reverse(
  xT=xT_256,
  T=T,
  betas=betas,
  posterior_betas=posterior_betas,
  alpha_bars=alpha_hats,
  model=model_256
).squeeze(1).detach().numpy()

reverse_time_256 = time.time() - reverse_time_256
print(f"Reverse process duration : { reverse_time_256:.2f} seconds")

synthetic_data_256 = inverse_standard(scaled_synthetic_data_256, train_snp500)
SYN_PATH_256 = dir / "data" / "syn_data_256.joblib"
joblib.dump(synthetic_data_256, SYN_PATH_256)

Reverse process duration : 185.82 seconds


['D:\\CodingHenry\\research_MBKM\\data\\syn_data_256.joblib']

In [10]:
reverse_time_512 = time.time()

scaled_synthetic_data_512 = reverse(
  xT=xT_512,
  T=T,
  betas=betas,
  posterior_betas=posterior_betas,
  alpha_bars=alpha_hats,
  model=model_512
).squeeze(1).detach().numpy()

reverse_time_512 = time.time() - reverse_time_512
print(f"Reverse process duration : { reverse_time_512:.2f} seconds")

synthetic_data_512 = inverse_standard(scaled_synthetic_data_512, train_snp500)
SYN_PATH_512 = dir / "data" / "syn_data_512.joblib"
joblib.dump(synthetic_data_512, SYN_PATH_512)

Reverse process duration : 373.63 seconds


['D:\\CodingHenry\\research_MBKM\\data\\syn_data_512.joblib']